# 融合分割代码

## 这个示例展示如何使用一个端到端融合感知网络做极端场景实例分割

调用接口：
- tianmoucv.proc.segmentation.TM_seg 


In [ ]:
%load_ext autoreload
# use %autoreload command to auto-reload all libraries

## 必要的包和工具

In [ ]:
%autoreload
from IPython.display import clear_output
import sys,time,cv2,torch,os,random,argparse
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

from tianmoucv.isp import lyncam_raw_comp,demosaicing_npy
from tianmoucv.data import TianmoucDataReader
from tianmoucv.proc.reconstruct import TianmoucRecon_tiny
from tianmoucv.proc.nn.utils import tdiff_split_sum,_COLORS,segmentation_classes,coco_segmentation_classes_80,draw_result
from tianmoucv.isp import SD2XY

def images_to_video(frame_list,name,size=(640,320),Flip=True):
    fps = 60        
    ftmax = max([np.max(ft) for ft in frame_list])
    ftmin = min([np.min(ft) for ft in frame_list])
    out = cv2.VideoWriter(name,0x7634706d , fps, size)
    for ft in frame_list:
        ft = (ft-ftmin)/(ftmax-ftmin)
        ft2 = (ft*255).astype(np.uint8)
        out.write(ft2)
    out.release()

## TM_seg分割网络调用示例

In [ ]:
%autoreload
from tianmoucv.proc.segmentation import TM_seg 

device = torch.device('cuda:0')
torch.cuda.set_device(device)

yolo_seg_tmc = TM_seg(ckpt_path = None,device = device,_optim=False)
# ckpt_path = url 或者 local
reconstructor = TianmoucRecon_tiny(ckpt_path=None,_optim=False).to(device)

## 准备数据

In [ ]:
train='/data/lyh/tianmoucData/tianmoucReconDataset/train/'
dirlist = os.listdir(train)
traindata = [train + e for e in dirlist]
val='/data/lyh/tianmoucData/tianmoucReconDataset/test/'
vallist = os.listdir(val)
valdata = [val + e for e in vallist]

key_list = [] #包含所有sample名作为匹配关键词
for sampleset in valdata:
    print(' ')
    print('---->',sampleset,'有：',len(os.listdir(sampleset)),'个样本--------------------')
    for e in os.listdir(sampleset):
        print(e,end=" ")
        key_list.append(e)
for sampleset in traindata:
    print(' ')
    print('---->',sampleset,'有：',len(os.listdir(sampleset)),'个样本--------------------')
    for e in os.listdir(sampleset):
        print(e,end=" ")
        key_list.append(e)     
        
all_data = valdata + traindata #包含所有数据的父路径的列表

## 执行推理过程

In [ ]:
%autoreload
%matplotlib inline 

key_list = ['underbridge_hdr_3']
for key in key_list:
    pathList = all_data
    # 读取sample列表
    dataset = TianmoucDataReader(pathList,showList=True,
                                 matchkey = key,
                                 ifcache = False,
                                 MAXLEN=-1,
                                 print_info=True)
    imlist = []
    start = 50
    end = start+30
    for index in range(start,end):
        rawsample = dataset[index]
        F0 = torch.Tensor(rawsample['F0'])
        F1 = torch.Tensor(rawsample['F1'])
        tsdiff_raw = rawsample['rawDiff']/255.0
        tsdiff = rawsample['tsdiff']

        td = tsdiff[0:1,...]
        sd = tsdiff[1:,...]

        w=640                                
        h=320
        biasw = (F0.shape[1]-w)//2
        biash = (F0.shape[0]-h)//2
        sample = dataset[index]
        sample['F0'] = torch.Tensor(sample['F0']).unsqueeze(0).permute(0,3,1,2)
        sample['F1'] = torch.Tensor(sample['F1']).unsqueeze(0).permute(0,3,1,2)
        sample['tsdiff'] = sample['tsdiff'].unsqueeze(0)
        # 重建一个视觉参考
        reconstructed_b = reconstructor(sample,
                                            w=640,
                                            h=320,
                                            bs=26,
                                            ifSingleDirection=False).float()
        reconstructed_b[reconstructed_b>1]=1
        reconstructed_b[reconstructed_b<0]=0

        # 对每一帧推理一次
        for t in range(sd.shape[1]-1):
            sd0 = sd[:,0,...]
            sdt = sd[:,t,...]

            td_sum = tdiff_split_sum(td[:,:t+1,...],cdim = 0,tdim=1) #sum on t dimension and concat on channel

            f0 = F0.permute(2,0,1)
            
            cat_input_full = torch.cat([f0,td_sum,sdt,sd0],dim=0).unsqueeze(0)
            cat_input_full = cat_input_full.to(yolo_seg_tmc.model.device) 

            #四张图，分别是原图和重建结果，用于绘制识别结果
            imshow0 = np.ascontiguousarray(f0.permute(1,2,0).numpy()*255).astype(np.uint8)
            imshow_full = np.ascontiguousarray(reconstructed_b[t,...].permute(1,2,0).cpu().numpy()*255).astype(np.uint8)

            #所有通道都输入
            preds_full,dt,mask_list_full = yolo_seg_tmc(cat_input_full,imshow_full,seg=True)
            pred_full = preds_full[0]

            #绘制结果
            imshow_result = imshow0.copy()
            draw_result(pred_full,imshow_result,mask_list_full,text='fusion inference')
            draw_result(pred_full,imshow_full,mask_list_full,text='map on VGT')

            cat_img = np.concatenate([imshow0[...,[2,1,0]],imshow_result[...,[2,1,0]],imshow_full[...,[2,1,0]]],axis=0)
            cat_img[cat_img>255]=255
            imlist.append(cat_img)
            
            if t in [0,12]:
                print('==time cost== inference: %f fps NMS: %f fps'% (1/dt[1],1/dt[2]))
                plt.figure(figsize=(8,4))
                plt.imshow(cat_img[...,[2,1,0]])
                plt.show()
                
        clear_output(wait=True)

## 保存视频

In [ ]:
images_to_video(imlist,'./'+key+'_det_result_without_rgb_result.mp4',size=(640,320*3),Flip=True)